# Tachiom Python API Demo

Covers the full `tachiom` Python interface:
- **Build** from raw `.npy` files (full pipeline or from pre-computed TAC)
- **Load** a previously saved index
- **Inspect** index properties
- **Search** — single query and batch
- **Evaluate** with `ir_measures`
- **Grid sweep** over retrieval parameters

Input files (LOTTE, `bench_tachiom_lotte_two_levels_pq.toml`):

| File | Shape | dtype | Description |
|---|---|---|---|
| `documents.npy` | `[N, dim]` | `f16` | Token vectors |
| `document_token_ids_flat.npy` | `[N]` | `i64`/`u32` | Token-type ID per token |
| `doclens.npy` | `[n_docs]` | `i32`/`i64` | Tokens per document |
| `queries.npy` | `[Q, n_tok, dim]` | `f32` | Query token vectors |
| `documents_ids.npy` *(opt)* | `[n_docs]` | any | Integer index → string doc ID |
| `queries_ids.npy` *(opt)* | `[Q]` | any | Integer index → string query ID |
| `centroids.npy` *(opt, TAC)* | `[K, dim]` | `f32` | Pre-computed coarse centroids |
| `assignments.npy` *(opt, TAC)* | `[N]` | `u32`/`u64` | Centroid assignment per token |

---
## 0b — Build the shared library

Run once per code change. `target-cpu=native` enables SIMD (AVX2/AVX-512) for PQ kernels.

In [ ]:
import subprocess, sys, os
from pathlib import Path

# Find the project root (directory containing Cargo.toml) regardless of where
# the notebook is opened from.
def _find_project_root():
    for p in [Path.cwd()] + list(Path.cwd().parents):
        if (p / "Cargo.toml").exists():
            return p
    raise RuntimeError("Could not find Cargo.toml — run this notebook from within the tachiom repo")

project_root = _find_project_root()

result = subprocess.run(
    ["maturin", "develop", "--release"],
    cwd=project_root,
    env={**os.environ, "RUSTFLAGS": "-C target-cpu=native"},
    capture_output=True,
    text=True,
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print(result.stderr[-2000:])
    raise RuntimeError("maturin build failed")
print("Build OK")

---
## 1 — Configuration

In [ ]:
from pathlib import Path

# ── Paths — update these to match your local setup ───────────────────────────
DATA_DIR   = Path("/path/to/dataset")
INDEX_PATH = Path("/path/to/indexes/tachiom/my_index")
QRELS_PATH = Path("/path/to/qrels.tsv")

VECTORS_FILE   = DATA_DIR / "documents.npy"
TOKEN_IDS_FILE = DATA_DIR / "document_token_ids_flat.npy"
DOCLENS_FILE   = DATA_DIR / "doclens.npy"
QUERIES_FILE   = DATA_DIR / "queries.npy"
DOC_IDS_FILE   = DATA_DIR / "documents_ids.npy"   # set to None to skip
QUERY_IDS_FILE = DATA_DIR / "queries_ids.npy"     # set to None to skip

# Only needed for Option B (build_from_tac)
CENTROIDS_FILE   = DATA_DIR / "centroids.npy"     # [K, dim] f32
ASSIGNMENTS_FILE = DATA_DIR / "assignments.npy"   # [N] u32/u64

# ── Build params ──────────────────────────────────────────────────────────────
BUILD_PARAMS = dict(
    total_centroids = 2_097_152,
    tac_n_iter      = 10,
    pq_sample_size  = 10_000_000,
    pq_n_iter       = 10,
    normalize       = True,
    pq_seed         = 42,
    hnsw_m          = 32,
    ef_construction = 1500,
)

# ── Search params ─────────────────────────────────────────────────────────────
K             = 10
SEARCH_PARAMS = dict(
    k_centroids     = 20,
    k_docs_to_score = 200,
    ef_search       = 30,
    alpha           = 0.4,
    beta            = None,
    lambda_         = None,
    num_threads     = 0,   # 0 = all cores; 1 = serial
)

EVAL_METRIC = "Success@5"

print("Config OK")

---
## 2 — Build / Load the index

Run **exactly one** of the three cells below.

### Option A.1 — Build from file paths (`build`)

Use when you have raw `.npy` files on disk and want the simplest call.

In [ ]:
import time, tachiom

t0 = time.perf_counter()
idx = tachiom.Tachiom.build(
    str(VECTORS_FILE),
    str(TOKEN_IDS_FILE),
    str(DOCLENS_FILE),
    **BUILD_PARAMS,
)
print(f"Built in {time.perf_counter() - t0:.2f}s")

INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)
idx.save(str(INDEX_PATH))
print(f"Saved to {INDEX_PATH}")

### Option A.2 — Build from numpy arrays (`build_from_arrays`)

Same full TAC → PQ → HNSW pipeline as A.1, but accepts numpy arrays instead of file paths.  
`vectors` is memory-mapped here — data is streamed from disk into the index with a single  
copy, so peak RAM during the build is the index itself rather than index + raw vectors.

In [ ]:
import numpy as np, time, tachiom

# vectors.npy stores raw f16 bit patterns as uint16 on disk.
# np.asarray() strips the memmap subclass so PyO3 accepts it; the Rust binding
# reinterprets the u16 bits as f16 — same as build() does internally.
vectors   = np.asarray(np.load(str(VECTORS_FILE),   mmap_mode='r'))                   # [N, dim]  u16, mmap
token_ids = np.asarray(np.load(str(TOKEN_IDS_FILE), mmap_mode='r')).astype(np.uint32) # [N]       u32
doclens   = np.asarray(np.load(str(DOCLENS_FILE),   mmap_mode='r')).astype(np.int32)  # [n_docs]  i32

print(f"vectors   : {vectors.shape}  dtype={vectors.dtype}  base is memmap={isinstance(vectors.base, np.memmap)}")
print(f"token_ids : {token_ids.shape}  dtype={token_ids.dtype}")
print(f"doclens   : {doclens.shape}  dtype={doclens.dtype}")

t0 = time.perf_counter()
idx = tachiom.Tachiom.build_from_arrays(
    vectors,
    token_ids,
    doclens,
    **BUILD_PARAMS,
)
print(f"Built in {time.perf_counter() - t0:.2f}s")

INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)
idx.save(str(INDEX_PATH))
print(f"Saved to {INDEX_PATH}")

### Option B — Build from pre-computed TAC centroids (skip clustering)

Use when you already have `centroids.npy` and `assignments.npy` from a previous TAC run.  
Runs PQ training + encoding from scratch but skips the k-means step.

In [ ]:
import time, tachiom

t0 = time.perf_counter()
idx = tachiom.Tachiom.build_from_tac(
    str(VECTORS_FILE),
    str(TOKEN_IDS_FILE),
    str(DOCLENS_FILE),
    str(CENTROIDS_FILE),
    str(ASSIGNMENTS_FILE),
    pq_sample_size  = BUILD_PARAMS["pq_sample_size"],
    pq_n_iter       = BUILD_PARAMS["pq_n_iter"],
    normalize       = BUILD_PARAMS["normalize"],
    pq_seed         = BUILD_PARAMS["pq_seed"],
    hnsw_m          = BUILD_PARAMS["hnsw_m"],
    ef_construction = BUILD_PARAMS["ef_construction"],
)
print(f"Built in {time.perf_counter() - t0:.2f}s")

INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)
idx.save(str(INDEX_PATH))
print(f"Saved to {INDEX_PATH}")

### Option C — Load a previously saved index

In [ ]:
import time, tachiom

t0 = time.perf_counter()
idx = tachiom.Tachiom.load(str(INDEX_PATH))
print(f"Loaded in {time.perf_counter() - t0:.2f}s")

---
## 3 — Index inspection

Properties: `len`, `dim`, `n_tokens`, `n_centroids`.  
Method: `print_space_usage()`.

In [ ]:
print(repr(idx))
print(f"Documents  : {idx.len:,}")
print(f"Dim        : {idx.dim}")
print(f"Tokens     : {idx.n_tokens:,}")
print(f"Centroids  : {idx.n_centroids:,}")
print()
idx.print_space_usage()

---
## 4 — Load queries

In [ ]:
import numpy as np

# shape (Q, n_tokens, dim), must be float32 C-contiguous
queries = np.ascontiguousarray(np.load(QUERIES_FILE).astype(np.float32))
assert queries.ndim == 3, f"Expected (Q, n_tokens, dim), got {queries.shape}"
assert queries.shape[2] == idx.dim, f"dim mismatch: {queries.shape[2]} vs {idx.dim}"

# Flat 2D view for batch_search — shares memory with queries, no copy.
# Uniform token count: (Q, n_tok, dim) → (Q * n_tok, dim).
tokens_flat = queries.reshape(-1, queries.shape[-1])

doc_ids_arr   = np.load(DOC_IDS_FILE,   allow_pickle=True) if DOC_IDS_FILE   and DOC_IDS_FILE.exists()   else None
query_ids_arr = np.load(QUERY_IDS_FILE, allow_pickle=True) if QUERY_IDS_FILE and QUERY_IDS_FILE.exists() else None

print(f"Queries     : {queries.shape}      (dtype={queries.dtype})")
print(f"tokens_flat : {tokens_flat.shape}  (dtype={tokens_flat.dtype}, shares buffer={np.shares_memory(queries, tokens_flat)})")
print(f"doc_ids loaded   : {doc_ids_arr is not None}")
print(f"query_ids loaded : {query_ids_arr is not None}")

---
## 5 — Single-query search

`idx.search(query, k, *, k_centroids, k_docs_to_score, ef_search, alpha, beta, lambda_)`  
Input: 2D `f32` array `(n_tokens, dim)`. Returns `(scores, doc_ids)` — both 1D, length `k`.

In [ ]:
import time

q = np.ascontiguousarray(queries[0])  # (n_tokens, dim)

t0 = time.perf_counter()
sc, di = idx.search(
    q, k=K,
    k_centroids     = SEARCH_PARAMS["k_centroids"],
    k_docs_to_score = SEARCH_PARAMS["k_docs_to_score"],
    ef_search       = SEARCH_PARAMS["ef_search"],
    alpha           = SEARCH_PARAMS["alpha"],
    beta            = SEARCH_PARAMS["beta"],
    lambda_         = SEARCH_PARAMS["lambda_"],
)
print(f"Latency: {(time.perf_counter() - t0)*1000:.1f} ms")
print("scores: [", end="")
for score in sc[:-1]:
    print(f"{score:.4f}, ", end="")
print(f"{sc[-1]:.4f}]")

print(f"doc_ids: [", end="")
for d_idx in di[:-1]:
    print(f"{d_idx}, ", end="")
print(f"{di[-1]}]")
print()

sentinel = np.iinfo(np.uint32).max
print("Top results for query 0:")
for rank, (score, d_idx) in enumerate(zip(sc, di), 1):
    if d_idx == sentinel:
        break
    label = str(doc_ids_arr[d_idx]) if doc_ids_arr is not None else str(d_idx)
    print(f"  {rank:2d}.  doc={label}  score={score:.4f}")

---
## 6 — Batch search

`idx.batch_search(tokens, n_queries, k, *, offsets, num_threads, k_centroids, k_docs_to_score, ef_search, alpha, beta, lambda_)`

Input: flat 2D `f32` array `(total_tokens, dim)` + `n_queries: int`.  
Returns `(scores, doc_ids)` — both 2D `(n_queries, k)`.

**Uniform mode** (`offsets=None`, used here): all queries have the same token count —  
`total_tokens` must be divisible by `n_queries`.

**Ragged mode** (`offsets=[n_queries+1]` u64): variable token count per query.

In [ ]:
import time

n_queries = len(queries)
t0 = time.perf_counter()
scores, doc_indices = idx.batch_search(tokens_flat, n_queries, k=K, **SEARCH_PARAMS)
elapsed_ms = (time.perf_counter() - t0) * 1000

print(f"Total : {elapsed_ms:.1f} ms   per query : {elapsed_ms / n_queries:.2f} ms")
print(f"scores      shape : {scores.shape}      dtype={scores.dtype}")
print(f"doc_indices shape : {doc_indices.shape}  dtype={doc_indices.dtype}")

---
## 7 — Evaluate with ir_measures

Builds a TREC run from the batch-search output and computes the configured metric.

In [ ]:
import pandas as pd, ir_measures

sentinel = np.iinfo(np.uint32).max
rows = []
for q_idx in range(n_queries):
    q_id = str(query_ids_arr[q_idx]) if query_ids_arr is not None else str(q_idx)
    for rank, (s, d) in enumerate(zip(scores[q_idx], doc_indices[q_idx]), 1):
        if d == sentinel:
            break
        d_id = str(doc_ids_arr[d]) if doc_ids_arr is not None else str(d)
        rows.append({"query_id": q_id, "doc_id": d_id, "rank": rank, "score": float(s)})

run_df = pd.DataFrame(rows)
print(f"{len(run_df)} rows in run")
run_df.head()

In [ ]:
qrels_df = pd.read_csv(QRELS_PATH, sep="\t",
                       names=["query_id", "useless", "doc_id", "relevance"])
if len(pd.unique(qrels_df["useless"])) != 1:  # 3-col format
    qrels_df = pd.read_csv(QRELS_PATH, sep="\t",
                           names=["query_id", "doc_id", "relevance", "useless"])

for df in (qrels_df, run_df):
    df["query_id"] = df["query_id"].astype(str)
    df["doc_id"]   = df["doc_id"].astype(str)

print(f"{len(qrels_df)} qrels rows")
qrels_df.head()

In [ ]:
metric = ir_measures.parse_measure(EVAL_METRIC)
result = ir_measures.calc_aggregate([metric], qrels_df, run_df)
print(f"{metric}: {result[metric]:.6f}")

---
## 8 — Parameter grid sweep

Three configs from the TOML (expected values are from the TOML comments — results may vary slightly due to RNG).

In [ ]:
import time, pandas as pd, ir_measures

metric = ir_measures.parse_measure(EVAL_METRIC)
sentinel = np.iinfo(np.uint32).max

configs = [
    dict(ef_search=30, k_centroids=20, k_docs_to_score=200, alpha=0.4),  # ≈ 67.5, 11 ms/q sequentially
    dict(ef_search=30, k_centroids=20, k_docs_to_score=300, alpha=0.4),  # ≈ 68.0, 14 ms/q sequentially
    dict(ef_search=40, k_centroids=25, k_docs_to_score=500, alpha=0.4),  # ≈ 68.5, 23 ms/q sequentially
]

rows_out = []
for cfg in configs:
    t0 = time.perf_counter()
    sc, di = idx.batch_search(tokens_flat, n_queries, k=K, num_threads=SEARCH_PARAMS["num_threads"], **cfg)
    ms_per_q = (time.perf_counter() - t0) * 1000 / n_queries

    rows_run = []
    for q_idx in range(n_queries):
        q_id = str(query_ids_arr[q_idx]) if query_ids_arr is not None else str(q_idx)
        for rank, (s, d) in enumerate(zip(sc[q_idx], di[q_idx]), 1):
            if d == sentinel:
                break
            d_id = str(doc_ids_arr[d]) if doc_ids_arr is not None else str(d)
            rows_run.append({"query_id": q_id, "doc_id": d_id, "rank": rank, "score": float(s)})
    run_cfg = pd.DataFrame(rows_run)
    run_cfg["query_id"] = run_cfg["query_id"].astype(str)
    run_cfg["doc_id"]   = run_cfg["doc_id"].astype(str)

    val = ir_measures.calc_aggregate([metric], qrels_df, run_cfg)[metric]
    rows_out.append({**cfg, EVAL_METRIC: round(val, 4), "ms/query": round(ms_per_q, 2)})
    print(f"{cfg}  {EVAL_METRIC}={val:.4f}  {ms_per_q:.2f} ms/q")

pd.DataFrame(rows_out)